# Поиск фильма через OMDb API
Ключ API хранится в файле `.env` (переменная `OMDB_API_KEY`), в репозиторий он не попадает.

In [10]:
# импорт библиотек
import json
import os

import requests
from dotenv import load_dotenv

## Настройки и загрузка ключа из .env

In [11]:
# настройки и загрузка ключа из .env
API_URL = "https://www.omdbapi.com/"
TIMEOUT_SECONDS = 10

load_dotenv()
API_KEY = os.getenv("OMDB_API_KEY")
if not API_KEY:
    raise ValueError("Ключ OMDB_API_KEY не найден в .env")

## Функция запроса к API

In [12]:
# запрос к OMDb API
def fetch_movie(title: str, api_key: str) -> dict:
    """Возвращает JSON-ответ OMDb в виде словаря.

    Бросает RuntimeError с понятным сообщением (без ключа в тексте).
    """
    try:
        response = requests.get(
            API_URL,
            params={"t": title, "apikey": api_key},
            timeout=TIMEOUT_SECONDS,
        )
    except requests.exceptions.Timeout:
        raise RuntimeError("Сервис не ответил вовремя. Попробуйте позже.") from None
    except requests.exceptions.ConnectionError:
        raise RuntimeError("Нет соединения с сервисом. Проверьте интернет.") from None
    except requests.exceptions.RequestException:
        raise RuntimeError("Не удалось выполнить запрос к сервису.") from None

    if response.status_code == 401:
        raise RuntimeError("Ошибка доступа: неверный или неактивированный API-ключ.")
    if not response.ok:
        raise RuntimeError(f"Сервис вернул ошибку HTTP {response.status_code}.")

    try:
        return response.json()
    except ValueError:
        raise RuntimeError("Сервис вернул ответ не в формате JSON.") from None

## Функция вывода результата

In [13]:
# вывод результата
def print_movie(data: dict) -> None:
    """Печатает нужные поля фильма или сообщение об ошибке."""
    if data.get("Response") == "False":
        print("Фильм не найден:", data.get("Error", "неизвестная ошибка"))
        return

    fields = [
        ("Название", "Title"),
        ("Год", "Year"),
        ("Жанр", "Genre"),
        ("Режиссёр", "Director"),
        ("Актёры", "Actors"),
        ("Рейтинг IMDb", "imdbRating"),
    ]
    for label, key in fields:
        print(f"{label}: {data.get(key, 'нет данных')}")

    print(
        f"\nФильм '{data.get('Title')}' ({data.get('Year')}) — "
        f"жанр: {data.get('Genre')}. Режиссёр: {data.get('Director')}."
    )

## Запуск

In [14]:
# запуск
title = input("Введите фильм для поиска: ").strip()

if not title:
    print("Название фильма не может быть пустым.")
else:
    try:
        json_data = fetch_movie(title, API_KEY)
    except RuntimeError as error:
        print("Ошибка:", error)
    else:
        # полный JSON, чтобы видеть все поля ответа
        print(json.dumps(json_data, ensure_ascii=False, indent=2))
        print("-" * 40)
        print_movie(json_data)

{
  "Title": "The Avengers",
  "Year": "2012",
  "Rated": "PG-13",
  "Released": "04 May 2012",
  "Runtime": "143 min",
  "Genre": "Action, Sci-Fi",
  "Director": "Joss Whedon",
  "Writer": "Joss Whedon, Zak Penn",
  "Actors": "Robert Downey Jr., Chris Evans, Scarlett Johansson",
  "Plot": "Earth's mightiest heroes must come together and learn to fight as a team if they are going to stop the mischievous Loki and his alien army from enslaving humanity.",
  "Language": "English, Russian",
  "Country": "United States",
  "Awards": "Nominated for 1 Oscar. 43 wins & 81 nominations total",
  "Poster": "https://m.media-amazon.com/images/M/MV5BNGE0YTVjNzUtNzJjOS00NGNlLTgxMzctZTY4YTE1Y2Y1ZTU4XkEyXkFqcGc@._V1_QL75_UX380_CR0,0,380,562_.jpg",
  "Ratings": [
    {
      "Source": "Internet Movie Database",
      "Value": "8.0/10"
    },
    {
      "Source": "Rotten Tomatoes",
      "Value": "91%"
    },
    {
      "Source": "Metacritic",
      "Value": "69/100"
    }
  ],
  "Metascore": "69",
  "